# D1: Monthly Report Generator

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Generate automated reports** from housing data
2. **Create multiple output formats** (HTML, JSON, PDF)
3. **Design report templates** for different audiences
4. **Schedule recurring analysis** for monthly updates

## Why This Matters

Data analysis is only valuable if it reaches decision-makers. This notebook shows how to:
- Transform raw data into professional reports
- Create reusable report templates
- Export data for dashboards and web applications
- Automate recurring reporting workflows

## Report Audiences

| Audience | Format | Content Focus |
|----------|--------|--------------|
| City Council | PDF | High-level summary |
| Planning Staff | HTML | Detailed metrics |
| Public | Web | Interactive maps |
| Developers | JSON | Raw data access |

---

## Overview

Generate automated monthly housing development reports.

**Report Contents:**
- New proposals
- New approvals
- Construction starts
- Completions

**Output Formats:**
- HTML
- PDF (via HTML)
- JSON (for dashboards)

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Current Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"Total units: {df['net_units'].sum():,.0f}")

## 3. Generate Report Data

In [ ]:
# Generate monthly report
if df is not None:
    report = generate_monthly_report(df, datetime.now())
    
    print(f"Report Month: {report['report_month']}")
    print(f"Generated: {report['generated_at']}")
    print("\nMetrics:")
    for key, value in report['metrics'].items():
        print(f"  {key}: {value}")

## 4. Current Pipeline Summary

In [ ]:
# Status summary
if df is not None:
    summary = generate_status_summary(df)
    
    print("Pipeline Summary:")
    print("="*60)
    print(f"Total Projects: {summary['total_projects']}")
    print(f"Total Units: {summary['total_units']:,}")
    
    print("\nBy Status:")
    for status, count in sorted(summary['by_status'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {status}: {count}")

## 5. Top Projects This Period

In [ ]:
# Largest projects currently in pipeline
if df is not None:
    print("Largest Active Projects:")
    print("="*60)
    
    top_projects = df.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']]
    display(top_projects)

## 6. Export HTML Report

In [ ]:
# Generate HTML report
if df is not None:
    timestamp = datetime.now().strftime('%Y%m')
    html_path = REPORTS_DIR / f'housing_report_{timestamp}.html'
    
    export_to_html(report, html_path)
    print(f"\nHTML report saved: {html_path}")

## 7. Export JSON for Dashboard

In [ ]:
# Export JSON for dashboard
if df is not None:
    json_path = REPORTS_DIR / f'housing_report_{timestamp}.json'
    export_to_json(report, json_path)
    print(f"JSON report saved: {json_path}")

## 8. Report Preview

In [ ]:
# Display report preview
from IPython.display import HTML, display

if html_path.exists():
    with open(html_path) as f:
        html_content = f.read()
    display(HTML(html_content))

---

## Summary

This notebook:
- Generated monthly housing development report
- Created pipeline summary
- Exported to HTML and JSON

**Next:** Run `D2_dashboard_data_export.ipynb` for dashboard data.